In [ ]:
# 내가만큰 코드가 최적화에서 잘 돌아가는지 확인하는 코드
# 소요시간 확인 용

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os


#  실행 환경 및 디바이스 설정
# CUDA(NVIDIA GPU) 사용 가능 여부를 확인합니다. MPS(Mac)는 이 예제에서 제외하고 CUDA 중심으로 설명합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")


if device.type == 'cuda':
    print(f"GPU 모델: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 코어 및 텐서 코어 활용 준비 완료.")


#  데이터셋 및 로더 준비 (최적화 적용)
# 가상의 대용량 데이터를 생성합니다. (10,0000개 샘플, 1,024개 특징)
input_size = 1024
num_samples = 100000
inputs = torch.randn(num_samples, input_size)
targets = torch.randint(0, 10, (num_samples,)) # 10개 클래스 분류
dataset = TensorDataset(inputs, targets)


# [최적화 포인트 1: pin_memory와 num_workers]
# pin_memory=True: 데이터를 페이지 고정 메모리에 할당하여 전송 가속화
# num_workers: 데이터 로딩을 병렬 프로세스로 처리 (CPU 코어 수 고려)
# 주의: Windows에서는 멀티프로세싱 방식 차이로 num_workers > 0 시 에러가 발생할 수 있으므로 안전장치 추가
num_workers = 4 if os.name!= 'nt' else 0


train_loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False,
    drop_last=True # 마지막 불완전한 배치를 버려 텐서 크기 재할당 오버헤드 방지
)


#  모델 정의
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, 10)
   
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


model = SimpleNet().to(device) # 모델을 GPU 메모리로 이동


criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)


#  최적화된 학습 루프
print(f"\n학습 시작... (num_workers={num_workers})")
model.train()


# 성능 측정을 위한 타이머
total_start_time = time.time()


for epoch in range(2): # 데모를 위해 2 에폭만 실행
    epoch_loss = 0.0
    batch_start_time = time.time()
   
    for i, (data, target) in enumerate(train_loader):
        # [최적화 포인트 2: 비동기 데이터 전송]
        # non_blocking=True를 사용하여 데이터 전송 중에 CPU가 멈추지 않고
        # 다음 명령(optimizer.zero_grad 등)을 미리 준비하도록 함.
        # 이는 pin_memory=True일 때만 실질적인 효과가 있습니다.
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
       
        optimizer.zero_grad() # 이전 기울기 초기화
       
        output = model(data) # 순전파 (Forward)
        loss = criterion(output, target) # 손실 계산
        loss.backward() # 역전파 (Backward)
        optimizer.step() # 가중치 업데이트
       
        # [최적화 포인트 3: item() 사용의 중요성]
        # loss는 계산 그래프가 포함된 Tensor 객체입니다.
        # 만약 `total_loss += loss`라고 쓰면, 그래프 전체가 계속 메모리에 누적되어 OOM(Out of Memory) 발생.
        # `.item()`을 호출하여 Python의 스칼라(숫자) 값만 추출해야 그래프 참조가 끊깁니다.
        epoch_loss += loss.item()
       
    epoch_duration = time.time() - batch_start_time
    print(f"Epoch {epoch+1} 완료 | 평균 손실: {epoch_loss / len(train_loader):.4f} | 소요 시간: {epoch_duration:.4f}초")


total_duration = time.time() - total_start_time
print(f"전체 학습 완료. 총 소요 시간: {total_duration:.4f}초")


#  평가 및 추론 시 최적화
print("\n추론(Inference) 테스트 중...")
model.eval() # 평가 모드 전환 (Dropout, BatchNorm 동작 변경)


# [최적화 포인트 4: torch.no_grad()]
# 추론 시에는 기울기(Gradient)를 계산할 필요가 없습니다.
# no_grad 컨텍스트를 사용하면 중간 연산 결과(Activation)를 저장하지 않아
# 메모리 사용량이 대폭 감소하고 속도가 빨라집니다.
with torch.no_grad():
    sample_input = torch.randn(10, input_size).to(device)
    prediction = model(sample_input)
    print("추론 완료. 출력 크기:", prediction.shape)


# [추가 팁: GPU 캐시 정리]
# 학습 중 OOM 발생 시 디버깅용으로만 사용하세요.
# 루프 내에서 매번 호출하면 동기화 오버헤드로 인해 속도가 매우 느려집니다.
torch.cuda.empty_cache()
print("GPU 캐시 정리 완료.")


현재 사용 중인 디바이스: cuda
GPU 모델: NVIDIA GeForce GTX 1660 SUPER
CUDA 코어 및 텐서 코어 활용 준비 완료.

학습 시작... (num_workers=4)
Epoch 1 완료 | 평균 손실: 2.3049 | 소요 시간: 2.7387초
Epoch 2 완료 | 평균 손실: 2.3030 | 소요 시간: 2.6341초
전체 학습 완료. 총 소요 시간: 5.3738초

추론(Inference) 테스트 중...
추론 완료. 출력 크기: torch.Size([10, 10])
GPU 캐시 정리 완료.


In [2]:
# 배치노멀라이즈 : 신경망 안정됨
# 블럭안에: Loss 까지만 존재 
import torch
import torch.nn as nn
import torch.optim as optim


#  AMP 준비
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cpu':
    print("주의: AMP는 GPU(CUDA) 환경에서 텐서 코어를 사용할 때 진정한 성능 향상이 있습니다.")
    # CPU에서는 bfloat16이 지원되지만 속도 향상은 CPU 아키텍처에 따라 다릅니다.


# 모델 생성 (기본적으로 FP32로 생성됨)
model = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.Linear(64 * 32 * 32, 10) # 예시를 위한 Flatten 생략 및 차원 맞춤 가정
).to(device)


# 실제 실행 가능한 형태의 모델로 수정 (Flatten 포함)
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(64 * 32 * 32, 10)
   
    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))
        x = self.flatten(x)
        return self.fc(x)


model = SimpleCNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()


# 입력 데이터 (배치 크기 64, 3채널, 32x32 이미지)
data = torch.randn(64, 3, 32, 32).to(device)
targets = torch.randint(0, 10, (64,)).to(device)


# [핵심 객체] GradScaler 초기화
# enabled=False로 설정하면 일반 FP32 학습과 동일하게 동작합니다 (디버깅 시 유용)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


print("AMP 학습 단계 시작...")


optimizer.zero_grad()


#  Autocast 컨텍스트 매니저 사용
# 이 블록 내부의 연산(Forward)은 PyTorch가 자동으로 판단하여
# - 안전한 연산(Conv, Linear 등)은 FP16으로
# - 민감한 연산(Softmax, Sum 등)은 FP32로 수행합니다.
dtype_to_use = torch.float16 if device.type == 'cuda' else torch.bfloat16
with torch.autocast(device_type=device.type, dtype=dtype_to_use):
    output = model(data)
    loss = loss_fn(output, targets)
    # loss는 FP32로 자동 캐스팅되어 나옵니다 (안정성 위해)


#  Scaled Backward (스케일링된 역전파)
# 일반적인 loss.backward() 대신 사용합니다.
# 내부적으로 loss * scale_factor 연산을 수행하여 기울기 언더플로우를 방지합니다.
scaler.scale(loss).backward()


#  Scaled Step (스케일링된 가중치 업데이트)
# optimizer.step() 대신 사용합니다.
# 1. 기울기를 scale_factor로 나눕니다 (Unscale).
# 2. 기울기에 Inf나 NaN이 있는지 검사합니다.
# 3. 문제가 없으면 가중치를 업데이트하고, 문제가 있으면 이번 배치는 건너뜁니다(Skip).
scaler.step(optimizer)


#  Scaler Update
# 다음 반복을 위해 scale_factor를 조정합니다.
# - 연속으로 성공하면 scale_factor를 2배로 늘려 정밀도 한계를 테스트합니다.
# - 실패(NaN/Inf 감지)하면 scale_factor를 절반으로 줄입니다.
scaler.update()


print(f"AMP 단계 완료. 현재 Scale Factor: {scaler.get_scale()}")
print(f"계산된 손실값: {loss.item():.4f}")


AMP 학습 단계 시작...
AMP 단계 완료. 현재 Scale Factor: 65536.0
계산된 손실값: 2.3278


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# 데이터 및 모델 준비
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = TensorDataset(torch.randn(1000, 10), torch.randn(1000, 1))
loader = DataLoader(dataset, batch_size=32) # 실제 메모리에 올리는 작은 배치
                                            # 파라미터도 같이 올라감
model = nn.Linear(10, 1).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1)


#  스케줄러 설정 (ReduceLROnPlateau)
# 'min' 모드: 모니터링하는 값(Loss)이 줄어들지 않을 때 동작
# Loss가 안좋을때 어떤 동작을 할래~~~~
# patience=2: 2 에폭 동안 성능 향상이 없으면 LR 감소
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)
# 코드 뜻 :  2에폭마다 성능이 안줄면 학습률을 줄이겠따
# <비교> RNN loss를 누적하면서 학습



#  그래디언트(기울기) 누적 설정
accumulation_steps = 4 # 32 * 4 = 128의 효과적인 배치 크기(Virtual Batch Size)를 가짐


print("학습 루프 시작...")
model.train()


for epoch in range(5):
    epoch_loss = 0.0
    optimizer.zero_grad() # 에폭 시작 시 초기화 (혹은 step 직후 초기화)
                          # 주의:zero_grad가 에폭에 있음 --> 에폭마다 초기화 하겠다는 뜻 즉, 평균 loss를 구하겠따.
                          # 합이 구해지니 평균이 필요함
   
    for i, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
       
        # 순전파
        output = model(data)
        loss = nn.MSELoss()(output, target)
       
        # [중요] Loss 정규화
        # 기울기는 더해지므로(Sum), 배치 크기가 커진 효과를 내려면
        # 누적 스텝 수로 나눠주어 평균(Mean)을 맞춰야 합니다.
        loss = loss / accumulation_steps
       
        loss.backward() # 기울기 누적됨 (기존 기울기에 더하기)
       
        # 누적 스텝을 만족할 때만 업데이트 수행
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()      # 가중치 업데이트
            optimizer.zero_grad() # 기울기 초기화
           

        # zero_grad 없으면 오차가 누적됨

        # 기록용 loss는 다시 스케일 복구
        epoch_loss += loss.item() * accumulation_steps


    avg_loss = epoch_loss / len(loader)
   
    # [중요] 스케줄러 업데이트 (Validation Loss를 넣어야 함, 여기선 Training Loss로 대체)
    # ReduceLROnPlateau는 반드시 메트릭(수치)을 인자로 받아야 합니다.
    scheduler.step(avg_loss)
   
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Current LR: {current_lr:.6f}")


학습 루프 시작...
Epoch 1, Loss: 1.0806, Current LR: 0.100000
Epoch 2, Loss: 1.0157, Current LR: 0.100000
Epoch 3, Loss: 1.0138, Current LR: 0.100000
Epoch 4, Loss: 1.0137, Current LR: 0.100000
Epoch 5, Loss: 1.0137, Current LR: 0.100000


In [6]:
import torch
import time
import os


# 복잡한 연산을 수행하는 모델 (컴파일 효과를 보기 위함)
def get_model():
    model = torch.nn.Sequential(
        torch.nn.Linear(512, 1024),
        torch.nn.ReLU(),
        torch.nn.Linear(1024, 2048),
        torch.nn.GELU(),
        torch.nn.Linear(2048, 512),
        torch.nn.Sigmoid()
    ).cuda()
    return model


# 성능 측정 함수
def benchmark(model, input_data, n_iters=50):
    # 워밍업: 초기 컴파일 및 캐싱 오버헤드를 제외하기 위해 몇 번 먼저 실행
    for _ in range(10):
        model(input_data)
    torch.cuda.synchronize() # GPU 연산 완료 대기 (정확한 시간 측정을 위해 필수)
   
    start = time.time()
    for _ in range(n_iters):
        model(input_data)
    torch.cuda.synchronize()
    end = time.time()
   
    return (end - start) / n_iters


if __name__ == "__main__":
    if not torch.cuda.is_available():
        print("이 예제는 GPU가 필요합니다.")
    else:
        if os.name == 'nt':
            print("알림: Windows에서 torch.compile은 베타 기능이며, C++ 컴파일러 설정이 필요할 수 있습니다.")


        input_data = torch.randn(64, 512).cuda()
       
        #  기본 모드 (Eager Mode)
        print("Eager Mode 벤치마크 중...")
        model_eager = get_model()
        t_eager = benchmark(model_eager, input_data)
        print(f"Eager Mode 평균 시간: {t_eager*1000:.3f} ms")
       
        #  컴파일 모드 (Compiled Mode)
        print("Compile Mode 벤치마크 중...")
        try:
            # 한 줄의 마법: 모델을 컴파일하여 최적화
            # mode='reduce-overhead': CUDA 그래프 등을 사용하여 CPU 오버헤드 최소화
            model_compiled = torch.compile(get_model(), mode="reduce-overhead")
            t_compiled = benchmark(model_compiled, input_data)
           
            print(f"Compiled Mode 평균 시간: {t_compiled*1000:.3f} ms")
            print(f"속도 향상: {t_eager / t_compiled:.2f}배")
           
        except Exception as e:
            print(f"\n컴파일 실패: {e}")
            print("해결책: Linux 환경을 사용하거나, Windows의 경우 MSVC 컴파일러를 설치해야 합니다.")


# 속도 진짜 차이많이남
# (도커)리눅스에서 돌아가서 깔끔하게 돌아간거임...쌩 window 였으면 오류 났음 -->MSVC 컴파일러를 설치해야 합니다.
# 컴파일러는 리눅스가 돌아가기 최적임

Eager Mode 벤치마크 중...
Eager Mode 평균 시간: 1.528 ms
Compile Mode 벤치마크 중...
Compiled Mode 평균 시간: 0.425 ms
속도 향상: 3.60배


In [ ]:
>